# Credit Union Quarterly Data Ingestion

Ingests NCUA call-report data quarter-by-quarter into a Delta table.
Uses `iter_quarter_dataframes()` to stream one quarter at a time — keeps peak
memory to a single quarter's DataFrame, and writes incrementally to Delta via
`spark.createDataFrame()` (gRPC, no local filesystem access needed).

In [ ]:
import re
from ingest_ncua_call_report import iter_quarter_dataframes

def clean_col(name):
    """Sanitize column name for Delta table compatibility."""
    name = name.lower()
    name = re.sub(r"[ ,;{}()\n\t=/]+", "_", name)
    name = name.replace("-", "_")
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def make_unique(cols):
    """Deduplicate column names by appending a counter."""
    seen = {}
    new_cols = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            new_cols.append(c)
        else:
            seen[c] += 1
            new_cols.append(f"{c}_{seen[c]}")
    return new_cols

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ncua")

In [ ]:
start_year = 1994
end_year = 2026
table_name = "workspace.ncua.ncua_bronze"
# Increase download_workers to download more quarters in parallel.
# Each batch downloads this many quarters concurrently, then processes
# and writes them one at a time before starting the next batch.
download_workers = 8
first_write = True

for quarter, pdf in iter_quarter_dataframes(
    start_year=start_year,
    end_year=end_year,
    download_workers=download_workers,
):
    # Cast all columns to string for schema consistency across decades
    pdf = pdf.astype(str)

    # Clean column names for Delta compatibility
    cleaned = [clean_col(c) for c in pdf.columns]
    pdf.columns = make_unique(cleaned)

    try:
        sdf = spark.createDataFrame(pdf)
        write_mode = "overwrite" if first_write else "append"
        (
            sdf.write
            .format("delta")
            .mode(write_mode)
            .option("mergeSchema", "true")
            .saveAsTable(table_name)
        )
        print(f"  -> {write_mode.upper()}: {len(pdf):,} rows for {quarter} written to {table_name}")
        first_write = False
    except Exception as e:
        print(f"  -> FAILED writing {quarter}: {e}")
    finally:
        del pdf, sdf

print("\nIngestion complete.")

In [ ]:
display(spark.sql("SELECT COUNT(*) as total_rows FROM workspace.ncua.ncua_bronze"))
display(spark.sql("SELECT DISTINCT cycle_date FROM workspace.ncua.ncua_bronze ORDER BY cycle_date"))